In [1]:
import os
import glob
import json
import torch
import torchvision.transforms as T
from PIL import Image
from fastai.vision.all import resnet50
from fastai.vision.learner import create_cnn_model

In [2]:
segmentation_dir = "../data/output/segmentation/"
model_path = "../data/models/resnet50_fish_safe.pth" 
output_nlp_dir = "../data/output/nlp_payload"
os.makedirs(output_nlp_dir, exist_ok=True)

In [5]:
checkpoint = torch.load(model_path, map_location=torch.device('cpu'), weights_only=False)
vocab = checkpoint['vocab']
weights = checkpoint['model_weights']

In [ ]:
model = create_cnn_model(resnet50, n_out=len(vocab), pretrained=False)
model.load_state_dict(weights)
model.eval() 

/Users/filimono/Documents/Inno/Interactive-fish-study-system/.venv/lib/python3.13/site-packages/fastai/vision/learner.py:297: UserWarning: `create_cnn_model` has been renamed to `create_vision_model` -- please update your code
  warn("`create_cnn_model` has been renamed to `create_vision_model` -- please update your code")


Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample

In [7]:
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [8]:
search_pattern = os.path.join(segmentation_dir, "*.jpg")
segmented_images = glob.glob(search_pattern)

In [ ]:
if not segmented_images:
    print(f"В папке {segmentation_dir} не найдено ни одного изображения.")
else:
    print(f"Найдено изображений от SAM 2: {len(segmented_images)}\n")
    
    for img_path in segmented_images:
        image_name = os.path.splitext(os.path.basename(img_path))[0]
        output_json_path = os.path.join(output_nlp_dir, f"{image_name}_nlp.json")
        print(f"Обработка: {image_name}...")
        
        img = Image.open(img_path).convert('RGB')
        tensor = transform(img).unsqueeze(0) 
        
        with torch.no_grad():
            preds = model(tensor)
            probs = torch.softmax(preds, dim=1)[0]
            pred_idx = torch.argmax(probs).item()
            
        confidence = probs[pred_idx].item()
        pred_class = vocab[pred_idx]
        
        nlp_data = {
            "detected_objects": [
                {
                    "class": pred_class,
                    "confidence": round(confidence, 2),
                    "source": "SAM 2 + ResNet-50", 
                    "image_path": img_path
                }
            ]
        }
        
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(nlp_data, f, indent=4, ensure_ascii=False)
            
        print(f"Готово! Распознано: {pred_class} (Уверенность: {confidence:.2f})")

print("\nВсе изображения успешно классифицированы!")

Найдено изображений от SAM 2: 10

Обработка: angel3_crop_20260710_153914...
Готово! Распознано: AngelFish (Уверенность: 1.00)
Обработка: angel3_full_20260710_153914...
Готово! Распознано: AngelFish (Уверенность: 1.00)
Обработка: blue3_crop_20260710_153124...
Готово! Распознано: BlueTang (Уверенность: 1.00)
Обработка: blue3_full_20260710_153124...
Готово! Распознано: BlueTang (Уверенность: 1.00)
Обработка: clownfish2_full_20260710_153535...
Готово! Распознано: ClownFish (Уверенность: 1.00)
Обработка: fly2_full_20260710_153621...
Готово! Распознано: ButterflyFish (Уверенность: 1.00)
Обработка: gold_full_20260710_154021...
Готово! Распознано: GoldFish (Уверенность: 0.99)
Обработка: gold_crop_20260710_154021...
Готово! Распознано: GoldFish (Уверенность: 1.00)
Обработка: fly2_crop_20260710_153621...
Готово! Распознано: ButterflyFish (Уверенность: 1.00)
Обработка: clownfish2_crop_20260710_153535...
Готово! Распознано: ClownFish (Уверенность: 1.00)

Все изображения успешно классифицированы!
